# 📊 Quant Finance Starter
## Stock Price Analysis & Moving Average Trading Strategy

This notebook covers:
1. Loading and exploring stock market data
2. Calculating moving averages and daily returns
3. Visualizing price trends
4. Implementing a simple moving average crossover strategy
5. Backtesting the strategy against buy-and-hold

In [ ]:
import os
import sys

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

# Add scripts directory to path so we can reuse helper functions
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, os.path.join(REPO_ROOT, 'scripts'))

DATA_PATH = os.path.join(REPO_ROOT, 'data', 'AAPL_stock_data.csv')
IMAGES_DIR = os.path.join(REPO_ROOT, 'images')
os.makedirs(IMAGES_DIR, exist_ok=True)

print('Setup complete.')

---
## 1. Load & Explore the Data

In [ ]:
df = pd.read_csv(DATA_PATH, parse_dates=['Date'], index_col='Date')
df.sort_index(inplace=True)

print(f'Shape: {df.shape}')
print(f'Period: {df.index[0].date()} → {df.index[-1].date()}')
df.head()

In [ ]:
df.describe()

---
## 2. Moving Averages

In [ ]:
SHORT_WINDOW = 20
LONG_WINDOW  = 50

df[f'SMA_{SHORT_WINDOW}'] = df['Close'].rolling(window=SHORT_WINDOW).mean()
df[f'SMA_{LONG_WINDOW}']  = df['Close'].rolling(window=LONG_WINDOW).mean()

fig, ax = plt.subplots()
ax.plot(df.index, df['Close'],              label='Close Price',       linewidth=1.2, color='#1f77b4')
ax.plot(df.index, df[f'SMA_{SHORT_WINDOW}'],label=f'{SHORT_WINDOW}-Day SMA', linewidth=1.5, color='orange', linestyle='--')
ax.plot(df.index, df[f'SMA_{LONG_WINDOW}'], label=f'{LONG_WINDOW}-Day SMA',  linewidth=1.5, color='red',    linestyle='--')
ax.set_title('AAPL – Close Price with Moving Averages', fontsize=13, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Price (USD)')
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
fig.autofmt_xdate()
fig.savefig(os.path.join(IMAGES_DIR, 'AAPL_price_moving_averages.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## 3. Daily Returns

In [ ]:
df['Daily_Return']     = df['Close'].pct_change() * 100
df['Cumulative_Return']= (1 + df['Close'].pct_change()).cumprod() - 1

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Daily returns time series
axes[0].plot(df.index, df['Daily_Return'], linewidth=0.8, color='#1f77b4')
axes[0].axhline(0, color='red', linewidth=0.8, linestyle='--')
axes[0].set_title('AAPL – Daily Returns (%)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Date')
axes[0].set_ylabel('Return (%)')
axes[0].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
axes[0].xaxis.set_major_locator(mdates.MonthLocator(interval=3))

# Histogram
returns_clean = df['Daily_Return'].dropna()
axes[1].hist(returns_clean, bins=40, color='#1f77b4', edgecolor='white', alpha=0.8)
axes[1].axvline(returns_clean.mean(), color='red', linestyle='--',
                label=f'Mean: {returns_clean.mean():.2f}%')
axes[1].set_title('Return Distribution', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Daily Return (%)')
axes[1].set_ylabel('Frequency')
axes[1].legend()

fig.autofmt_xdate()
fig.tight_layout()
fig.savefig(os.path.join(IMAGES_DIR, 'AAPL_daily_returns.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f'Average daily return : {returns_clean.mean():.4f}%')
print(f'Daily volatility     : {returns_clean.std():.4f}%')
print(f'Total return         : {df["Cumulative_Return"].iloc[-1]*100:.2f}%')

---
## 4. Moving Average Crossover Strategy

In [ ]:
# Signal: 1 = long, 0 = flat
df['Signal']   = 0
df.loc[df[f'SMA_{SHORT_WINDOW}'] > df[f'SMA_{LONG_WINDOW}'], 'Signal'] = 1
df['Position'] = df['Signal'].diff()

buys  = df[df['Position'] ==  1]
sells = df[df['Position'] == -1]

print(f'Buy signals : {len(buys)}')
print(f'Sell signals: {len(sells)}')

---
## 5. Backtesting

In [ ]:
INITIAL_CAPITAL = 10_000.0

shares_held = 0
cash        = INITIAL_CAPITAL
holdings_list = []
cash_list     = []

for _, row in df.iterrows():
    if row['Position'] == 1:       # BUY
        shares_held = int(cash // row['Close'])
        cash -= shares_held * row['Close']
    elif row['Position'] == -1:    # SELL
        cash += shares_held * row['Close']
        shares_held = 0
    holdings_list.append(shares_held * row['Close'])
    cash_list.append(cash)

portfolio = pd.DataFrame({'Close': df['Close'],
                           'Holdings': holdings_list,
                           'Cash': cash_list}, index=df.index)
portfolio['Total']   = portfolio['Holdings'] + portfolio['Cash']
portfolio['Returns'] = portfolio['Total'].pct_change()

# Buy-and-hold benchmark
bh_shares    = int(INITIAL_CAPITAL // df['Close'].iloc[0])
bh_cash_left = INITIAL_CAPITAL - bh_shares * df['Close'].iloc[0]
bh_value     = bh_shares * df['Close'] + bh_cash_left

# Performance metrics
strat_return = (portfolio['Total'].iloc[-1] - INITIAL_CAPITAL) / INITIAL_CAPITAL * 100
bh_return    = (bh_value.iloc[-1]           - INITIAL_CAPITAL) / INITIAL_CAPITAL * 100
daily_ret    = portfolio['Returns'].dropna()
sharpe       = daily_ret.mean() / daily_ret.std() * np.sqrt(252) if daily_ret.std() > 0 else 0
max_dd       = ((portfolio['Total'] - portfolio['Total'].cummax()) / portfolio['Total'].cummax() * 100).min()

print(f'Strategy Total Return : {strat_return:.2f}%')
print(f'Buy & Hold Return     : {bh_return:.2f}%')
print(f'Sharpe Ratio          : {sharpe:.3f}')
print(f'Max Drawdown          : {max_dd:.2f}%')
print(f'Final Portfolio Value : ${portfolio["Total"].iloc[-1]:,.2f}')

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

# Panel 1: Price + signals
axes[0].plot(df.index, df['Close'],               label='Close Price',              linewidth=1,   color='#1f77b4')
axes[0].plot(df.index, df[f'SMA_{SHORT_WINDOW}'], label=f'{SHORT_WINDOW}-Day SMA',  linewidth=1.3, color='orange', linestyle='--')
axes[0].plot(df.index, df[f'SMA_{LONG_WINDOW}'],  label=f'{LONG_WINDOW}-Day SMA',   linewidth=1.3, color='red',    linestyle='--')
axes[0].scatter(buys.index,  buys['Close'],  marker='^', color='green', s=80, label='Buy Signal',  zorder=5)
axes[0].scatter(sells.index, sells['Close'], marker='v', color='red',   s=80, label='Sell Signal', zorder=5)
axes[0].set_title('AAPL – Moving Average Crossover Strategy', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Price (USD)')
axes[0].legend(loc='upper left', fontsize=8)

# Panel 2: Equity curves
axes[1].plot(portfolio.index, portfolio['Total'], label='Strategy Portfolio', linewidth=1.5, color='green')
axes[1].plot(df.index,        bh_value,           label='Buy & Hold',         linewidth=1.5, color='#1f77b4', linestyle='--')
axes[1].axhline(INITIAL_CAPITAL, color='gray', linestyle=':', linewidth=0.8)
axes[1].set_title('Portfolio Value Comparison', fontsize=12)
axes[1].set_xlabel('Date')
axes[1].set_ylabel('Portfolio Value (USD)')
axes[1].legend()

axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
axes[1].xaxis.set_major_locator(mdates.MonthLocator(interval=3))
fig.autofmt_xdate()
fig.tight_layout()
fig.savefig(os.path.join(IMAGES_DIR, 'AAPL_strategy_backtest.png'), dpi=150, bbox_inches='tight')
plt.show()